In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv('OPENAI_API_KEY')
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')
os.environ['TAVILY_API_KEY']=os.getenv('TAVILY_API_KEY')
os.environ['GEMINI_API_KEY']=os.getenv('GEMINI_API_KEY')


In [15]:
import langchain
langchain.debug = True

In [6]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

agent = create_deep_agent(model='google_genai:gemini-3.5-flash')

agent2 = create_deep_agent(
    model='google_genai:gemini-3.5-flash',
    backend=StateBackend()
)

In [7]:
result = agent2.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it."
        )
    }]
})
print("\n--- Agent reply -------------------------------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
[{'type': 'text', 'text': 'I have created the file at `/notes/todo.txt` with the following content:\n\n1. Record video\n2. Edit video\n3. Upload video', 'extras': {'signature': 'EvMFCvAFAQw51sfAfbbPis6MwGYXfrNejmr4RKKYhipUbmQFsZBXmU5/WuKg3ffb8hd3rmF/5xeIQPrtPd8o+Bo5XQLJYSadfGcGl9f8yoq0MgQhacspeV+5Gw/dgN0bQviDJw315zWlGs8w5gxLa45LF0Ahn2G7sFemtgws72qdlEDQrU8pr8A8rKGmOYQ1Nd197jCFPM1nYP6f5y+qMpv+dyVhArenFWzMfNFu6EFi1infDBnrhgp+PZ8Q2ERqIexlItJaqUtX2qDOjAChEdPZeXyUV3UhzGgiAeaYy0GC6gZ+IYrw38QF22QTjdK7ldNe6/oaKI1SpJilPuxW8pX8Q9FJu3lEVyBu/w2s4eSHxU4ADddlWXUi9T8srl5EjE19/TJoC1JDTMJsIiaFwp3FnLy71dwfbF39+2ZaImHRXjO22nNYQ5W8lL7jbguK+YK945TuaBZ7aBS8zUs78tBIKcxp4xEVvmihGBpmbWT2oB2VUQ7E8QFJtxINfkj5pI7jpUDyKeo7hxH/KrFag1/89JaVZyiGWVy+G1mvp4IU84ylke97kwrKesRAn2c0THaAbD3nmAZsMXeWL/fvifbIWsZ3BoOZVsZ5QY3BYZ6fUV8opxu3VNNXBXPUGKjVO6BOw5azu2E4B/ywifFlxr1ceXcuC+mMH/TbVnsMJrZLKLv7hCBaBv6Mmi+1akW8hMoecHTiEFUUPVRbVqPL+U+MmXkRoCzwBgkAQ+KtnNX+4hFZ8mS

In [8]:
result

{'messages': [HumanMessage(content="Create a file at /notes/todo.txt with exactly this content:\n1. Record video\n2. Edit video\n3. Upload video\nThen tell me you've done it.", additional_kwargs={}, response_metadata={}, id='ba8bdb8e-fc37-4153-ae25-6dbe8c9fb3ac'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'write_file', 'arguments': '{"content": "1. Record video\\n2. Edit video\\n3. Upload video\\n", "file_path": "/notes/todo.txt"}'}, '__gemini_function_call_thought_signatures__': {'m494nkca': 'EvEFCu4FAQw51senni/t/8oPsPyBvO55zHKt8yHIZMapSD3saJ6CUGtBj6e53f4wSrH52F+SAwAnKhWCIxmAO9XhEQr8OfavZgR5zbag12eipHAHdA+fJbujk00+EbTdj55atWvhuZI7XPr07JoWAZ3GeLGvDyZZkT7RlFjKqCkGAigAniF9Q+AvrMhoXq6tiHklVlrmY23jHecfSSCZM3han5+lA6dycc6BQCr+QmIlbMykBcqo9TC+qGmBIPqAzJoEdMHy8wAx0j8opcUxi/aNU2i3TTnq/tJHgfL99sqcsRs00VQUZplPahF6ccqfi5s26INsQtiuyNobQeXjeVWATcAYXx2VlFoXTStzSREpoBYjn3TjJXKi7tdTJyRGwCuCjIwq0MDI+DfwRhuG8xv/2fRgrysXC2M7K9OqxwMA3m9ijqqpBUFeVGMPNGVG2bQyaj0tR7/X/a4rRWSTi1c3EA

In [9]:
print("\n--- Backend check -----------------------------------------------")
files = result.get("files", {})

if files:
    print(f"✅ StateBackend is working — {len(files)} file(s) in state:")
    for path, content in files.items():
        print(f"\n📄 {path}\n{'-' * 40}\n{content}")
else:
    print("⚠️  No files found in state. Either the agent didn't write a file, "
          "or the backend isn't wired up correctly.")


--- Backend check -----------------------------------------------
✅ StateBackend is working — 1 file(s) in state:

📄 /notes/todo.txt
----------------------------------------
{'content': '1. Record video\n2. Edit video\n3. Upload video\n', 'encoding': 'utf-8', 'created_at': '2026-06-26T16:45:17.675001+00:00', 'modified_at': '2026-06-26T16:45:17.675001+00:00'}


In [10]:
followup = agent2.invoke({
    # carry forward prior messages + the files state
    "messages": result["messages"] + [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me ."
    }],
    "files": result.get("files", {}),   # <-- pass the virtual filesystem along
})

print("\n--- Read-back (same thread) -------------------------------------")
print(followup["messages"][-1].content)


--- Read-back (same thread) -------------------------------------
[{'type': 'text', 'text': 'Here is the content of `/notes/todo.txt`:\n\n1. Record video\n2. Edit video\n3. Upload video', 'extras': {'signature': 'ErwBCrkBAQw51sdoW5RszQ7nqQSRAHJJd86hdYC5gTg2ZKMcfTX2p8xMpr5s6BnGlcLkW82qU+ehYLnTPygeivZ0OKCOiR3p8raY+JZdx5p/zuUYyLBc3tcl9jzA92+UpLK77H0Fy+n02TVsC+tJ6OTkcnvxDcqCjaoLlFzT2FrTxOnEcUAhF5JVaecNVTpt+uyfUJfVYyXr4GpuD1kXi2d6Lx7KGcXXCfcVNP3NuUYAKxRjxcWlPjILmSKUht4='}}]


In [ ]:
from deepagents.backends import FilesystemBackend
from deepagents.
ROOT = '.'

agent = create_deep_agent(model='google_genai:gemini-3.5-flash', backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True))

print(f"✅ Agent created with FilesystemBackend(root_dir={ROOT!r}).")
print("   Files written by the agent will appear on your ACTUAL disk.")

✅ Agent created with FilesystemBackend(root_dir='.').
   Files written by the agent will appear on your ACTUAL disk.


In [12]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it."
        )
    }]
})

print("\n--- Agent reply -------------------------------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
[{'type': 'text', 'text': 'I have created the file at `/notes/todo.txt` with the specified content.', 'extras': {'signature': 'Er0CCroCAQw51sdasbNOKBQi3MZhvxNl15r7E2FSXq4mfe7Xn0h1RW6IW7YvSBPVvwXcsZATA5IkcmRkx5cuTT8ZbaZVk62Pmzc88ZeOiHryCCkpqV4szNUir/O57+uuYIEO9jY2lqRqqYg6v4dOyMRwjICDHL6sXhSZWSU7VylpL+MCOfOj4/8hAD+e844Kr0zJtNg9Tw3FtNis6ekDIslVdJ4fS/kB75Grqjmm+Krk5Y39uTjPuMnxCc1icyxeUyBCUPziGNeXrJnthJa6ueAQ9TkL+rz1AjR4MK5aAUyGOzG76bvEZUKve746LSid/0EXowB2ZOkAwREgMUakNr8HYdYi/HFSNu7sAZJAPSWHyeUGJZ0R+VFnXTlSv2Vb0WOU7HTr9jgWDlZaeFNIZwDZ1eC0OAJixWiKPl3jx50='}}]


In [13]:
from pathlib import Path
print("\n--- Backend check (real filesystem) -----------------------------")
disk_path = Path(ROOT) / "notes" / "todo.txt"

if disk_path.exists():
    print(f"✅ FilesystemBackend is working — file exists on disk:")
    print(f"📄 {disk_path.resolve()}\n{'-' * 40}")
    print(disk_path.read_text())
else:
    print(f"⚠️  Expected file not found at {disk_path.resolve()}")
    print("    The agent may not have called the write tool, or the path "
          "mapping differs.")



--- Backend check (real filesystem) -----------------------------
✅ FilesystemBackend is working — file exists on disk:
📄 C:\Projects\DeepAgent\deepagentsdemo\notes\todo.txt
----------------------------------------
1. Record video
2. Edit video
3. Upload video



In [16]:
fresh_agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
)

followup = fresh_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]
    # NOTE: no `files` state passed in — the file is read straight from disk
})

print("\n--- Read-back with a FRESH agent (proves disk persistence) ------")
print(followup["messages"][-1].content)


--- Read-back with a FRESH agent (proves disk persistence) ------
[{'type': 'text', 'text': '1. Record video\n2. Edit video\n3. Upload video', 'extras': {'signature': 'EtoBCtcBAQw51sdQ/okGmnpn1dv8GxaWm2L+b0rQ5KocgFQcVAFFtECmnV/hGww4RkW+i26GnpfN09y+M31cdr4YCjtFFl4T1VHMhs5hIbX2khL6Y9CYPMwVk+lFWOYXB4PIk5CznKw/ZS/wqn0E90fM6kGvuBnpVeW9oOxRzfIrp34G9MsX3bPuSQbiXPEo2sJ+cQ3HMRCW1w8D6Gd8NktH8Q6Y+7KQK2YKcsN5yDRLeG2ilLZDsXsep8TlRGh6jQnm7zY/siMJx5m4EoiGlUBEmqylEPceMzlCRu4='}}]


In [18]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend
store = InMemoryStore()

agent = create_deep_agent(
    model="openai:gpt-5.4",
    backend=StoreBackend(
        # Local dev: static namespace. No deployment runtime needed.
        # In a LangSmith Deployment you'd use the user-identity version.
        namespace=lambda rt: ("demo-user",),
    ),
    store=store,
)


print("✅ Agent created with StoreBackend + static namespace.")

✅ Agent created with StoreBackend + static namespace.
